In [1]:
# 📦 Requirements:
# pip install openai pandas scikit-learn numpy openpyxl tqdm faiss-cpu

import pandas as pd
import numpy as np
import re
import os
import json
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from openai import OpenAI
from tqdm import tqdm

# ---------- CONFIG ----------
API_KEY = os.getenv("OPENAI_API_KEY", "")
client = OpenAI(api_key=API_KEY) if API_KEY else None
GPT_MODEL = "gpt-4.1"
EMBEDDING_MODEL = "text-embedding-3-large"
output_file = "matches_output_with_explainability.xlsx"

# Methodology controls (strict 8-step pipeline)
USE_OPENAI_EMBEDDINGS = True
ENABLE_GPT_VALIDATION = True
GPT_TOP_N_PER_OPPORTUNITY = 3
STRICT_SECTOR_FILTER = True
SECTOR_MIN_SIMILARITY_TO_PASS = 0.20
SOFT_MATCH_MIN_SEMANTIC = 0.08
BATCH_SIZE = 10

# Scoring weights
W_PROFILE = 0.35
W_PRODUCT = 0.35
W_SECTOR = 0.20
W_GPT = 0.10

STOPWORDS = {
    "the", "and", "for", "with", "from", "into", "that", "this", "are", "was",
    "were", "will", "can", "their", "its", "our", "your", "about", "over", "under",
    "what", "when", "where", "who", "why", "how", "is", "of", "to", "in", "on", "a", "an"
}

# Generic business words that create noisy, non-informative semantic overlap.
GENERIC_BUSINESS_TERMS = {
    "advanced", "company", "companies", "group", "global", "regional", "international",
    "development", "develop", "developing", "solution", "solutions", "service", "services",
    "market", "markets", "business", "industry", "industries", "portfolio", "capabilities",
    "operations", "operational", "based", "including", "providing", "provide", "provides",
    "supported", "support", "value", "quality", "efficient", "efficiency", "strategic",
    "innovation", "innovative", "growth", "project", "projects", "equipment", "systems",
    "technology", "technologies", "products", "product", "clients", "customer", "customers",
    "cities", "which", "specific", "future", "life", "safe", "driving", "smart", "digital",
    "integrated", "operating", "sustainable", "heavy", "flow", "provision", "process"
}

# Learned from the actual corpus at runtime (high-frequency tokens are de-prioritized).
DYNAMIC_COMMON_TOKENS = set()
TOKEN_IDF = {}
DOMAIN_KEYWORDS = set()
COMMON_TOKEN_DOC_RATIO_THRESHOLD = 0.12
MIN_COMMON_TOKEN_DOCS = 5

# Generic words frequently seen in text but weak for fit explainability.
EXPLANATION_BLACKLIST = {
    "ensuring", "generation", "provide", "providing", "develop", "developed", "development",
    "improve", "improving", "support", "supported", "operating", "process", "value", "quality",
    "needs", "tailored", "local", "near", "large", "used", "city", "global", "regional"
}

SECTOR_GROUPS = [
    {"ict", "hardware", "software", "electronics", "digital", "semiconductor", "telecom"},
    {"medtech", "medical", "healthcare", "biotech", "pharmaceutical", "pharma"},
    {"manufacturing", "industrial", "factory", "engineering", "construction", "materials"},
    {"energy", "power", "renewable", "oil", "gas", "utilities"},
] 

SECTOR_TOKEN_MAP = {
    "information": "ict",
    "communication": "ict",
    "communications": "ict",
    "telecommunications": "telecom",
    "electrical": "electronics",
    "electronic": "electronics",
    "semiconductors": "semiconductor",
    "manufacture": "manufacturing",
    "industrials": "industrial",
    "pharmaceuticals": "pharma",
    "medicine": "medical",
    "biologics": "biotech",
    "health": "healthcare",
}

# Sector ontology expansion to capture related sub-domains.
SECTOR_ONTOLOGY = {
    "ict": {"telecom", "network", "fiber", "datacenter", "server", "cloud", "embedded", "electronic", "electronics", "hardware", "software", "semiconductor"},
    "telecom": {"5g", "smallcell", "macro", "baseband", "antenna", "ran", "backhaul"},
    "medical": {"medtech", "diagnostic", "imaging", "clinical", "device", "devices", "sterile"},
    "pharma": {"pharmaceutical", "biologics", "api", "formulation", "fillfinish", "gmp"},
    "manufacturing": {"assembly", "fabrication", "machining", "tooling", "automation", "quality", "line"},
    "industrial": {"automation", "controls", "scada", "instrumentation", "process", "maintenance"},
    "energy": {"grid", "storage", "renewable", "solar", "wind", "utility", "power"},
}

BRIDGE_TERMS = {
    "industrial_to_ict": {
        "electronics", "electrical", "component", "components", "assembly", "hardware",
        "cables", "wiring", "automation", "control", "sensor", "sensors", "pcb", "device", "devices"
    },
    "industrial_to_medical": {
        "precision", "diagnostic", "imaging", "sterile", "medical", "healthcare",
        "cleanroom", "instrument", "instruments", "biotech", "pharma"
    },
    "ict_to_medical": {
        "medical", "healthcare", "diagnostic", "imaging", "device", "devices", "data", "software"
    },
}

# ---------- HELPERS ----------
def preprocess(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def tokenize_static(text):
    cleaned = preprocess(text)
    tokens = [
        t for t in cleaned.split()
        if t
        and t not in STOPWORDS
        and t not in GENERIC_BUSINESS_TERMS
        and len(t) > 2
        and not t.isdigit()
    ]
    return set(tokens)


def tokenize(text):
    tokens = tokenize_static(text)
    return {t for t in tokens if t not in DYNAMIC_COMMON_TOKENS}


def pick_existing_file(candidates):
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
        data_candidate = os.path.join("Data", candidate)
        if os.path.exists(data_candidate):
            return data_candidate
    raise FileNotFoundError(f"None of these files were found: {candidates}")


def normalize_company_columns(df):
    rename_map = {
        "Company Name": "company_name",
        "company name": "company_name",
        "company_name": "company_name",
        "Company Profile": "company_profile",
        "company profile": "company_profile",
        "company_profile": "company_profile",
        "Product/Services": "product and Services",
        "product/services": "product and Services",
        "Product and Services": "product and Services",
        "product and services": "product and Services",
        "product and Services": "product and Services",
    }
    df = df.rename(columns=rename_map)

    required_cols = ["company_name", "company_profile", "product and Services", "Sector"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required company columns: {missing}. Found: {list(df.columns)}")
    return df


def normalize_opportunity_columns(df):
    required_cols = [
        "What is the opportunity name?",
        "What is the opportunity description?",
        "What are the investment highlights?",
        "What is the value proposition of this opportunity?",
        "What are the key demand drivers?",
        "What materials are involved or required in the project?",
        "Sector",
    ]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required opportunity columns: {missing}. Found: {list(df.columns)}")
    return df


def canonicalize_tokens(tokens):
    return {SECTOR_TOKEN_MAP.get(t, t) for t in tokens}


def expand_sector_tokens(tokens):
    expanded = set(tokens)
    for t in list(tokens):
        if t in SECTOR_ONTOLOGY:
            expanded |= SECTOR_ONTOLOGY[t]
    return expanded


def sector_bridge_score(c_tokens, o_tokens, capability_tokens):
    # Cross-sector bridges based on capability terms in company profile/products.
    if (("industrial" in c_tokens or "manufacturing" in c_tokens) and ("ict" in o_tokens or "hardware" in o_tokens)) or (("industrial" in o_tokens or "manufacturing" in o_tokens) and ("ict" in c_tokens or "hardware" in c_tokens)):
        overlap = sorted(list(capability_tokens & BRIDGE_TERMS["industrial_to_ict"]))
        if overlap:
            return 0.58, "Moderate", f"Cross-sector bridge (Industrial ↔ ICT) via capabilities: {', '.join(overlap[:6])}."

    if (("industrial" in c_tokens or "manufacturing" in c_tokens) and ("medtech" in o_tokens or "medical" in o_tokens or "pharma" in o_tokens or "biotech" in o_tokens)) or (("industrial" in o_tokens or "manufacturing" in o_tokens) and ("medtech" in c_tokens or "medical" in c_tokens or "pharma" in c_tokens or "biotech" in c_tokens)):
        overlap = sorted(list(capability_tokens & BRIDGE_TERMS["industrial_to_medical"]))
        if overlap:
            return 0.48, "Weak", f"Cross-sector bridge (Industrial ↔ Medical/Pharma) via capabilities: {', '.join(overlap[:6])}."

    if (("ict" in c_tokens or "hardware" in c_tokens) and ("medtech" in o_tokens or "medical" in o_tokens or "pharma" in o_tokens)) or (("ict" in o_tokens or "hardware" in o_tokens) and ("medtech" in c_tokens or "medical" in c_tokens or "pharma" in c_tokens)):
        overlap = sorted(list(capability_tokens & BRIDGE_TERMS["ict_to_medical"]))
        if overlap:
            return 0.52, "Moderate", f"Cross-sector bridge (ICT ↔ Medical) via capabilities: {', '.join(overlap[:6])}."

    return 0.0, "No", "No capability-based cross-sector bridge identified."


def sector_similarity(company_sector, opportunity_sector, capability_text=""):
    c_tokens = expand_sector_tokens(canonicalize_tokens(tokenize(company_sector)))
    o_tokens = expand_sector_tokens(canonicalize_tokens(tokenize(opportunity_sector)))
    capability_tokens = expand_sector_tokens(canonicalize_tokens(tokenize(capability_text)))

    if not c_tokens or not o_tokens:
        return 0.0, "Unknown", "Sector data missing for company or opportunity."

    if preprocess(company_sector) == preprocess(opportunity_sector):
        return 1.0, "Exact", f"Exact sector match: '{company_sector}'."

    overlap = c_tokens & o_tokens
    union = c_tokens | o_tokens
    jaccard = len(overlap) / len(union) if union else 0.0

    group_hit = False
    for group in SECTOR_GROUPS:
        if (c_tokens & group) and (o_tokens & group):
            group_hit = True
            break

    score = max(jaccard, 0.75 if group_hit else 0.0)
    label = "No"
    reason = f"No clear sector overlap between '{company_sector}' and '{opportunity_sector}'."

    if score >= 0.80:
        label = "Strong"
    elif score >= 0.50:
        label = "Moderate"
    elif score > 0:
        label = "Weak"

    if overlap:
        reason = f"Shared sector terms: {', '.join(sorted(list(overlap))[:5])}."
    elif group_hit:
        reason = "Sector families align (same broader industry group)."

    if score < 0.50:
        bridge_score, bridge_label, bridge_reason = sector_bridge_score(c_tokens, o_tokens, capability_tokens)
        if bridge_score > score:
            score, label, reason = bridge_score, bridge_label, bridge_reason

    return score, label, reason


def is_informative_overlap_token(token):
    if not token or len(token) < 4:
        return False
    if token in EXPLANATION_BLACKLIST:
        return False
    # Strict mode: only domain vocabulary is allowed in explanations.
    return token in DOMAIN_KEYWORDS


def top_overlap_terms(text_a, text_b, limit=5):
    overlap = list(tokenize(text_a) & tokenize(text_b))
    informative = [t for t in overlap if is_informative_overlap_token(t)]

    # If empty, keep only domain terms with at least moderate specificity.
    if not informative:
        informative = [
            t for t in overlap
            if t in DOMAIN_KEYWORDS and TOKEN_IDF.get(t, 0.0) >= 1.6 and t not in EXPLANATION_BLACKLIST
        ]

    informative.sort(key=lambda t: (TOKEN_IDF.get(t, 0.0), len(t)), reverse=True)
    return informative[:limit]


def build_reason_from_overlap(overlap_terms, fallback_text):
    if overlap_terms:
        return f"Shared keywords: {', '.join(overlap_terms)}."
    return fallback_text


def semantic_score_band(score):
    if score >= 0.20:
        return "Very Strong"
    if score >= 0.12:
        return "Strong"
    if score >= 0.08:
        return "Moderate"
    if score >= 0.04:
        return "Weak"
    return "Very Weak"


def semantic_match_meaning(company_profile, company_products, opportunity_requirement, sim_profile, sim_product):
    profile_terms = top_overlap_terms(company_profile, opportunity_requirement, limit=7)
    product_terms = top_overlap_terms(company_products, opportunity_requirement, limit=7)
    overall = (sim_profile + sim_product) / 2.0

    meaning = (
        f"Semantic alignment is {semantic_score_band(overall)} "
        f"(profile={sim_profile:.3f}, product={sim_product:.3f})."
    )

    if profile_terms:
        meaning += f" Profile concepts in common: {', '.join(profile_terms)}."
    else:
        meaning += " Profile concepts overlap is limited."

    if product_terms:
        meaning += f" Product/service concepts in common: {', '.join(product_terms)}."
    else:
        meaning += " Product/service concepts overlap is limited."

    return meaning


def derive_ai_decision(sector_similarity, profile_similarity, product_similarity, ai_score, gpt_decision=None):
    """
    Business decision labels:
    - High Fit: strong evidence and ready for active pursuit
    - Good Fit: positive evidence, pursue with normal qualification
    - Review Needed: mixed signals, needs analyst review before pitch
    - Low Fit: weak evidence or explicit GPT rejection
    """
    semantic_avg = 0.5 * profile_similarity + 0.5 * product_similarity
    evidence = 0.45 * sector_similarity + 0.30 * profile_similarity + 0.25 * product_similarity

    if gpt_decision == "No":
        return "Low Fit"

    if gpt_decision == "Yes":
        if evidence >= 0.42 or ai_score >= 0.75:
            return "High Fit"
        return "Good Fit"

    if evidence >= 0.52 and semantic_avg >= 0.08:
        return "High Fit"
    if evidence >= 0.34 and semantic_avg >= 0.05:
        return "Good Fit"
    if evidence >= 0.22:
        return "Review Needed"
    return "Low Fit"


def build_ai_explanation(company_name, opportunity_name, ai_decision, sector_reason, semantic_meaning, profile_reason, product_reason, validation_status, gpt_text=None):
    explanation = (
        f"Decision: {ai_decision} for {company_name} on '{opportunity_name}'. "
        f"Why: {sector_reason} {semantic_meaning} "
        f"Profile evidence: {profile_reason} "
        f"Product evidence: {product_reason} "
        f"Validation: {validation_status}."
    )

    if gpt_text:
        explanation += f" LLM confirmation: {gpt_text}"
    return explanation


def build_ai_insight(company_name, opportunity_name, ai_decision, sector_reason):
    return (
        f"{company_name} is classified as '{ai_decision}' for '{opportunity_name}'. "
        f"Primary fit signal: {sector_reason}"
    )


def build_suggested_plan(company_name, opportunity_name, ai_decision, profile_terms, product_terms):
    profile_focus = ", ".join(profile_terms[:4]) if profile_terms else "core profile strengths"
    product_focus = ", ".join(product_terms[:4]) if product_terms else "delivery-relevant products/services"

    if ai_decision == "High Fit":
        next_step = "Move directly to proposal and stakeholder meetings"
    elif ai_decision == "Good Fit":
        next_step = "Run a short qualification session, then submit proposal"
    elif ai_decision == "Review Needed":
        next_step = "Perform targeted validation before client-facing commitment"
    else:
        next_step = "Keep as low priority unless strategic considerations require follow-up"

    return (
        f"{next_step}. Position {company_name} using profile strengths in {profile_focus}. "
        f"Anchor solution design around {product_focus}. Confirm compliance, timeline, and execution model before final commitment."
    )


def build_match_reason(sector_reason, profile_reason, product_reason, ai_decision):
    return (
        f"{ai_decision}. Sector: {sector_reason} Profile: {profile_reason} Product: {product_reason}"
    )


def build_embeddings(companies_df, opportunities_df):
    global DYNAMIC_COMMON_TOKENS, TOKEN_IDF, DOMAIN_KEYWORDS

    corpus = (
        companies_df["combined"].astype(str).tolist()
        + companies_df["products_clean"].astype(str).tolist()
        + opportunities_df["requirement"].astype(str).tolist()
    )

    # Build a domain vocabulary for explainability filtering.
    domain_seed = set()
    for k, v in SECTOR_ONTOLOGY.items():
        domain_seed.add(k)
        domain_seed |= set(v)
    for vals in BRIDGE_TERMS.values():
        domain_seed |= set(vals)

    # Keep domain vocabulary anchored to technical/sector fields only.
    for col in ["Sector", "What materials are involved or required in the project?"]:
        if col in opportunities_df.columns:
            for val in opportunities_df[col].astype(str).tolist():
                domain_seed |= tokenize_static(val)

    DOMAIN_KEYWORDS = {t for t in domain_seed if t and len(t) >= 4 and t not in GENERIC_BUSINESS_TERMS and t not in EXPLANATION_BLACKLIST}

    # Learn high-frequency generic tokens from this specific corpus.
    token_doc_counts = {}
    total_docs = max(1, len(corpus))
    for text in corpus:
        for tok in tokenize_static(text):
            token_doc_counts[tok] = token_doc_counts.get(tok, 0) + 1

    DYNAMIC_COMMON_TOKENS = {
        tok
        for tok, cnt in token_doc_counts.items()
        if cnt >= MIN_COMMON_TOKEN_DOCS and (cnt / total_docs) >= COMMON_TOKEN_DOC_RATIO_THRESHOLD
    }

    def embed_many(texts, model):
        vectors = []
        for t in texts:
            r = client.embeddings.create(model=model, input=[str(t)])
            vectors.append(r.data[0].embedding)
        return vectors

    companies_df = companies_df.copy()
    opportunities_df = opportunities_df.copy()

    if USE_OPENAI_EMBEDDINGS and client is not None:
        try:
            companies_df["profile_embedding"] = embed_many(companies_df["combined"].tolist(), EMBEDDING_MODEL)
            companies_df["product_embedding"] = embed_many(companies_df["products_clean"].tolist(), EMBEDDING_MODEL)
            opportunities_df["embedding"] = embed_many(opportunities_df["requirement"].tolist(), EMBEDDING_MODEL)

            # Build IDF table for explanation term ranking even when using API embeddings.
            tfidf_for_idf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
            tfidf_for_idf.fit(corpus)
            TOKEN_IDF = {
                term: float(idf)
                for term, idf in zip(tfidf_for_idf.get_feature_names_out(), tfidf_for_idf.idf_)
                if " " not in term
            }
            return companies_df, opportunities_df
        except Exception as e:
            print(f"OpenAI embeddings unavailable ({e}); falling back to TF-IDF embeddings.")

    # Fallback mode
    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    tfidf.fit(corpus)

    TOKEN_IDF = {
        term: float(idf)
        for term, idf in zip(tfidf.get_feature_names_out(), tfidf.idf_)
        if " " not in term
    }

    companies_df["profile_embedding"] = list(tfidf.transform(companies_df["combined"].astype(str)).toarray())
    companies_df["product_embedding"] = list(tfidf.transform(companies_df["products_clean"].astype(str)).toarray())
    opportunities_df["embedding"] = list(tfidf.transform(opportunities_df["requirement"].astype(str)).toarray())
    return companies_df, opportunities_df


def compute_similarity(vec1, vec2):
    return float(cosine_similarity([vec1], [vec2])[0][0])


def parse_json_safely(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        return None


def gpt_validate_and_explain(company_row, opportunity_row):
    if client is None:
        return "Not Run", 0.0, "OPENAI_API_KEY is not set."

    prompt = f"""
Evaluate company-opportunity fit and return strict JSON:
{{
  "decision": "Yes or No",
  "confidence": 0.0 to 1.0,
  "explanation": "2-4 sentence explanation focused on sector, profile and product fit"
}}

Company Name: {company_row['company_name']}
Company Sector: {company_row.get('Sector', '')}
Company Profile: {company_row['company_profile']}
Products/Services: {company_row['product and Services']}

Opportunity Name: {opportunity_row['What is the opportunity name?']}
Opportunity Sector: {opportunity_row.get('Sector', '')}
Opportunity Description: {opportunity_row['What is the opportunity description?']}
Opportunity Highlights: {opportunity_row['What are the investment highlights?']}
Demand Drivers: {opportunity_row['What are the key demand drivers?']}
Required Materials: {opportunity_row['What materials are involved or required in the project?']}
"""

    try:
        last_error = None
        response = None
        for model_name in [GPT_MODEL, "gpt-4o", "gpt-4o-mini"]:
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    temperature=0,
                    messages=[
                        {"role": "system", "content": "You are a strict industrial matching analyst."},
                        {"role": "user", "content": prompt}
                    ]
                )
                break
            except Exception as e:
                last_error = e

        if response is None:
            raise RuntimeError(f"No available GPT model for validation: {last_error}")

        content = response.choices[0].message.content.strip()
        parsed = parse_json_safely(content)

        if not parsed:
            decision = "Yes" if content.lower().startswith("yes") else "No"
            score = 0.8 if decision == "Yes" else 0.2
            return decision, score, content

        decision = str(parsed.get("decision", "No")).strip().title()
        decision = "Yes" if decision.startswith("Yes") else "No"
        score = float(parsed.get("confidence", 0.0))
        score = max(0.0, min(1.0, score))
        explanation = str(parsed.get("explanation", "No explanation provided.")).strip()
        return decision, score, explanation

    except Exception:
        return "Not Run", 0.0, "LLM validation unavailable; using rule-based scoring only."


def build_opportunity_requirement(row):
    fields = [
        row.get("What is the opportunity name?", ""),
        row.get("What is the opportunity description?", ""),
        row.get("What are the investment highlights?", ""),
        row.get("What is the value proposition of this opportunity?", ""),
        row.get("What are the key demand drivers?", ""),
        row.get("What materials are involved or required in the project?", ""),
        row.get("Who are the key players in this sector or project?", ""),
        row.get("Market data", ""),
        row.get("Risks and mitigations", ""),
    ]
    return preprocess(" ".join(str(x) for x in fields))


def match_entities(entities_a, entities_b, direction_label):
    print(f"🔍 Building raw matches for {direction_label}...")
    rows = []

    for src_idx, (_, a) in enumerate(entities_a.iterrows()):
        for _, b in entities_b.iterrows():
            if direction_label == "Company → Opportunity":
                comp = a
                opp = b
            else:
                comp = b
                opp = a

            sim_profile = compute_similarity(comp["profile_embedding"], opp["embedding"])
            sim_product = compute_similarity(comp["product_embedding"], opp["embedding"])

            capability_text = f"{comp['company_profile']} {comp['product and Services']}"
            sector_score, sector_label, sector_reason = sector_similarity(
                comp.get("Sector", ""), opp.get("Sector", ""), capability_text=capability_text
            )

            profile_overlap = top_overlap_terms(comp["company_profile"], opp["requirement"], limit=5)
            product_overlap = top_overlap_terms(comp["product and Services"], opp["requirement"], limit=5)

            profile_reason_core = build_reason_from_overlap(
                profile_overlap,
                "Profile has limited keyword overlap with opportunity requirements."
            )
            product_reason_core = build_reason_from_overlap(
                product_overlap,
                "Products/services have limited keyword overlap with opportunity requirements."
            )
            profile_reason = (
                f"Semantic profile similarity is {sim_profile:.3f} "
                f"({semantic_score_band(sim_profile)}). {profile_reason_core}"
            )
            product_reason = (
                f"Semantic product/service similarity is {sim_product:.3f} "
                f"({semantic_score_band(sim_product)}). {product_reason_core}"
            )
            semantic_meaning = semantic_match_meaning(
                comp["company_profile"],
                comp["product and Services"],
                opp["requirement"],
                sim_profile,
                sim_product,
            )

            # Step 7: Soft Match Mode (sector relaxed only when domain evidence exists).
            domain_overlap_terms = sorted(set(profile_overlap + product_overlap))
            has_domain_evidence = len(domain_overlap_terms) > 0
            if (
                sector_label == "No"
                and has_domain_evidence
                and (sim_product >= SOFT_MATCH_MIN_SEMANTIC or sim_profile >= SOFT_MATCH_MIN_SEMANTIC)
            ):
                boosted_score = max(sector_score, 0.35)
                if boosted_score > sector_score:
                    sector_score = boosted_score
                    sector_label = "Weak"
                    sector_reason = (
                        "Sector labels differ, but there is limited domain capability overlap "
                        f"({', '.join(domain_overlap_terms[:4])}) with moderate semantic support "
                        f"(profile={sim_profile:.3f}, product={sim_product:.3f})."
                    )
            elif sector_label == "No" and not has_domain_evidence:
                sector_reason = (
                    "Sector labels differ and no concrete domain overlap was found; "
                    f"semantic closeness alone is not treated as evidence (profile={sim_profile:.3f}, product={sim_product:.3f})."
                )

            # Step 3: Strict Sector Filtering gate before scoring/ranking.
            if STRICT_SECTOR_FILTER and sector_score < SECTOR_MIN_SIMILARITY_TO_PASS:
                continue

            ai_proxy_score = 0.50 * sector_score + 0.25 * sim_profile + 0.25 * sim_product
            base_score = (
                W_PROFILE * sim_profile
                + W_PRODUCT * sim_product
                + W_SECTOR * sector_score
                + W_GPT * ai_proxy_score
            )

            initial_decision = derive_ai_decision(
                sector_score,
                sim_profile,
                sim_product,
                ai_proxy_score,
                gpt_decision=None,
            )
            rule_explanation = build_ai_explanation(
                comp["company_name"],
                opp["What is the opportunity name?"],
                initial_decision,
                sector_reason,
                semantic_meaning,
                profile_reason,
                product_reason,
                validation_status="Rule-based screening (GPT not yet run)",
            )

            ai_insight = build_ai_insight(
                comp["company_name"],
                opp["What is the opportunity name?"],
                initial_decision,
                sector_reason,
            )
            suggested_plan = build_suggested_plan(
                comp["company_name"],
                opp["What is the opportunity name?"],
                initial_decision,
                profile_overlap,
                product_overlap,
            )
            match_reason = build_match_reason(sector_reason, profile_reason, product_reason, initial_decision)

            rows.append({
                "__src_idx": src_idx,
                "match_type": direction_label,
                "opportunity": opp["What is the opportunity name?"],
                "company": comp["company_name"],
                "company_sector": comp.get("Sector", ""),
                "opportunity_sector": opp.get("Sector", ""),
                "sector_similarity": round(sector_score, 3),
                "profile_similarity": round(sim_profile, 3),
                "product_similarity": round(sim_product, 3),
                "ai_score": round(ai_proxy_score, 3),
                "ai_decision": initial_decision,
                "final_score": round(base_score, 3),
                "Sector Match Reason": sector_reason,
                "Profile Match Reason": profile_reason,
                "Product Match Reason": product_reason,
                "ai_explanation": rule_explanation,
                "ai_insight": ai_insight,
                "suggested_plan": suggested_plan,
                "match_reason": match_reason,
            })

    df = pd.DataFrame(rows)
    df["rank"] = (
        df.groupby("opportunity")["final_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    if ENABLE_GPT_VALIDATION:
        print(f"🤖 Running GPT validation for top {GPT_TOP_N_PER_OPPORTUNITY} matches per opportunity...")
        validate_mask = df["rank"] <= GPT_TOP_N_PER_OPPORTUNITY
        validate_indices = list(df[validate_mask].index)

        for idx in tqdm(validate_indices, desc=f"GPT {direction_label}"):
            row = df.loc[idx]
            comp_row = companies[companies["company_name"] == row["company"]].iloc[0]
            opp_row = opportunities[opportunities["What is the opportunity name?"] == row["opportunity"]].iloc[0]

            decision, gpt_score, gpt_expl = gpt_validate_and_explain(comp_row, opp_row)
            decision_label = derive_ai_decision(
                df.at[idx, "sector_similarity"],
                df.at[idx, "profile_similarity"],
                df.at[idx, "product_similarity"],
                gpt_score,
                gpt_decision=decision,
            )

            df.at[idx, "ai_decision"] = decision_label
            df.at[idx, "ai_score"] = round(gpt_score, 3)
            df.at[idx, "ai_explanation"] = build_ai_explanation(
                row["company"],
                row["opportunity"],
                decision_label,
                df.at[idx, "Sector Match Reason"],
                "Semantic evidence preserved in profile and product reasons.",
                df.at[idx, "Profile Match Reason"],
                df.at[idx, "Product Match Reason"],
                validation_status="GPT validated",
                gpt_text=gpt_expl,
            )
            df.at[idx, "ai_insight"] = build_ai_insight(
                row["company"],
                row["opportunity"],
                decision_label,
                df.at[idx, "Sector Match Reason"],
            )
            df.at[idx, "suggested_plan"] = build_suggested_plan(
                row["company"],
                row["opportunity"],
                decision_label,
                [],
                [],
            )
            df.at[idx, "match_reason"] = build_match_reason(
                df.at[idx, "Sector Match Reason"],
                df.at[idx, "Profile Match Reason"],
                df.at[idx, "Product Match Reason"],
                decision_label,
            )

    df["final_score"] = (
        W_PROFILE * df["profile_similarity"]
        + W_PRODUCT * df["product_similarity"]
        + W_SECTOR * df["sector_similarity"]
        + W_GPT * df["ai_score"]
    ).round(3)

    df["rank"] = (
        df.groupby("opportunity")["final_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    output_cols = [
        "match_type",
        "opportunity",
        "company",
        "company_sector",
        "opportunity_sector",
        "sector_similarity",
        "profile_similarity",
        "product_similarity",
        "ai_score",
        "ai_decision",
        "final_score",
        "Sector Match Reason",
        "Profile Match Reason",
        "Product Match Reason",
        "ai_explanation",
        "rank",
        "ai_insight",
        "suggested_plan",
        "match_reason",
    ]

    total = len(entities_a)
    for start in range(0, total, BATCH_SIZE):
        end = min(start + BATCH_SIZE, total)
        batch_df = df[(df["__src_idx"] >= start) & (df["__src_idx"] < end)][output_cols].copy()
        batch_file = output_file.replace(
            ".xlsx",
            f"_{direction_label.replace(' → ', '_').replace(' ', '')}_batch_{start}_{end}.xlsx",
        )
        batch_df.to_excel(batch_file, index=False)
        print(f"✅ Saved {batch_file} with {len(batch_df)} matches.")


# ---------- MAIN ----------
company_file = pick_existing_file(["Companies.xlsx", "companies.xlsx"])
opportunity_file = pick_existing_file(["new_opportunities.xlsx", "New_Opportunities.xlsx"])

print("🔁 Loading files...")
companies = pd.read_excel(company_file)
opportunities = pd.read_excel(opportunity_file)
companies = normalize_company_columns(companies)
opportunities = normalize_opportunity_columns(opportunities)

print("🧹 Preprocessing and enrichment...")
companies["combined"] = companies[["company_name", "company_profile", "product and Services"]].astype(str).agg(" ".join, axis=1).apply(preprocess)
companies["products_clean"] = companies["product and Services"].astype(str).apply(preprocess)
opportunities["requirement"] = opportunities.apply(build_opportunity_requirement, axis=1)

print("🔄 Embedding...")
companies, opportunities = build_embeddings(companies, opportunities)

print("🚀 Running Company → Opportunity Matching...")
match_entities(companies, opportunities, "Company → Opportunity")

print("🚀 Running Opportunity → Company Matching...")
match_entities(opportunities, companies, "Opportunity → Company")

print("🎉 Matching complete.")

# ---------- EMAIL ENRICHMENT FOR INVESTOR FILE ----------
# Uses the same OpenAI key already configured above:
# API_KEY = os.getenv("OPENAI_API_KEY", "")
# client = OpenAI(api_key=API_KEY) if API_KEY else None

EMAIL_REGEX = re.compile(r"([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})")


def normalize_email(value):
    if pd.isna(value):
        return ""
    email = str(value).strip().strip(".,;:()[]<>\"'").lower()
    return email if re.match(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$", email) else ""


def first_email_in_text(text):
    if pd.isna(text):
        return ""
    match = EMAIL_REGEX.search(str(text))
    return normalize_email(match.group(1)) if match else ""


def detect_email_col(df):
    for col in df.columns:
        key = str(col).strip().lower()
        if key in {"email", "e-mail", "mail", "email address", "work email"}:
            return col
    return "Email"


def row_context_text(row):
    preferred = [
        "Investor Name", "Name", "Company", "Organization", "Website", "LinkedIn",
        "Profile", "Description", "Bio", "File Name", "Filename"
    ]
    context = []
    lower_map = {str(c).strip().lower(): c for c in row.index}

    for label in preferred:
        col = lower_map.get(label.lower())
        if col is not None and pd.notna(row.get(col)) and str(row.get(col)).strip():
            context.append(f"{label}: {row.get(col)}")

    if not context:
        for col, val in row.items():
            if pd.notna(val) and str(val).strip():
                context.append(f"{col}: {val}")

    return "\n".join(context)


def ask_openai_email(context_text, model="gpt-4o-mini"):
    if not API_KEY or client is None:
        return ""

    prompt = f"""
You are given one investor/company record.
Return only a reliable business email if confidently inferable from the context.
If no reliable email is available, return empty string.

Return strict JSON only:
{{"email":"<email_or_empty>"}}

Context:
{context_text}
""".strip()

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "Extract only reliable email. Do not hallucinate."},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
        )
        content = (response.choices[0].message.content or "").strip()
        data = json.loads(content)
        return normalize_email(data.get("email", ""))
    except Exception:
        fallback = first_email_in_text(locals().get("content", ""))
        return fallback


def add_emails_to_investor_file(
    input_path="Investors_Profiles_7th_March_V.xlsx",
    output_path="Investors_Profiles_7th_March_V_with_emails.xlsx",
    sheet_name=None,
    use_openai=True,
    model="gpt-4o-mini",
):
    sheet_arg = sheet_name if sheet_name is not None else 0
    df = pd.read_excel(input_path, sheet_name=sheet_arg)
    if isinstance(df, dict):
        if not df:
            raise ValueError(f"No sheets found in workbook: {input_path}")
        first_sheet_name = next(iter(df.keys()))
        df = df[first_sheet_name].copy()
        print(f"Loaded first sheet: {first_sheet_name}")

    email_col = detect_email_col(df)
    if email_col not in df.columns:
        df[email_col] = ""
    df[email_col] = df[email_col].fillna("").astype(str)

    regex_filled = 0
    ai_filled = 0

    for idx, row in df.iterrows():
        existing = normalize_email(row.get(email_col, ""))
        if existing:
            continue

        context_text = row_context_text(row)

        extracted = first_email_in_text(context_text)
        if extracted:
            df.at[idx, email_col] = extracted
            regex_filled += 1
            continue

        if use_openai and API_KEY and client is not None:
            inferred = ask_openai_email(context_text, model=model)
            if inferred:
                df.at[idx, email_col] = inferred
                ai_filled += 1

    df.to_excel(output_path, index=False)
    print(f"Saved: {output_path}")
    print(f"Email column: {email_col}")
    print(f"Filled by regex: {regex_filled}")
    print(f"Filled by OpenAI: {ai_filled}")

# Example usage:
# add_emails_to_investor_file(
#     input_path="Investors_Profiles_7th_March_V.xlsx",
#     output_path="Investors_Profiles_7th_March_V_with_emails.xlsx",
#     use_openai=True,
# )


🔁 Loading files...
🧹 Preprocessing and enrichment...
🔄 Embedding...


🚀 Running Company → Opportunity Matching...
🔍 Building raw matches for Company → Opportunity...


🤖 Running GPT validation for top 3 matches per opportunity...


GPT Company → Opportunity:   0%|          | 0/36 [00:00<?, ?it/s]

GPT Company → Opportunity:   3%|▎         | 1/36 [00:03<01:51,  3.18s/it]

...

GPT Opportunity → Company:  94%|█████████▍| 34/36 [01:16<00:04,  2.24s/it]

GPT Opportunity → Company:  97%|█████████▋| 35/36 [01:18<00:02,  2.24s/it]

GPT Opportunity → Company: 100%|██████████| 36/36 [01:21<00:00,  2.24s/it]

GPT Opportunity → Company: 100%|██████████| 36/36 [01:21<00:00,  2.25s/it]

✅ Saved matches_output_with_explainability_Opportunity_Company_batch_0_10.xlsx with 117 matches.
✅ Saved matches_output_with_explainability_Opportunity_Company_batch_10_12.xlsx with 16 matches.
🎉 Matching complete.
